# Joint Zero-Shot Motion Estimation and Iterative Reconstruction for Motion-Corrected Cardiac T1 Mapping

Cardiac T1 mapping suffers from cardiac motion: the standard clinical strategy is to
synchronize the acquisition to a fixed cardiac phase (ECG-triggering), which limits data
collection to a small fraction of each heartbeat and thus leads to long scan times and low
scan efficiency. Here we instead consider data acquired **continuously over the full cardiac
cycle**, which is far more efficient but requires estimating and correcting the complete
non-rigid cardiac deformation from highly undersampled data.

The proposed method obtains motion-corrected T1 maps by **coupling motion estimation, image
reconstruction and parametric fitting in a single iterative scheme**, instead of solving them
sequentially.

This notebook is a clean, self-contained walkthrough of the method implemented in
[`main.py`](../scripts/main.py). It runs the full pipeline on the simulated XCAT
phantom and ends with a plot of the estimated motion-corrected T1 map next to the reference
true map.

---

## Outline

0. [Method](#0) — problem formulation and proposed joint optimization
1. [Setup](#1) — imports, device, parameters
2. [Simulated data](#2) — XCAT phantom, k-space simulation
3. [Operators and initial reconstruction](#3) — $\mathbf{x}_\text{m}$
4. [T1 estimator and No MoCo baseline](#4) — effect of uncorrected motion
5. [Motion-compensated reference](#5) — $\mathbf{x}_\text{r}$, $\mathbf{p}_\text{r}$
6. [Motion estimation](#6) — initial motion fields $\boldsymbol{\phi}, \boldsymbol{\phi}^{-1}$
7. [Initialization (iteration 0)](#7) — $\mathbf{x}^{(0)}$, $\mathbf{p}^{(0)}$, SVD basis $\mathbf{U}$
8. [Alternating optimization loop](#8)
9. [Final T1 map](#9)

---

<a id="0"></a>
## 0. Method

### Problem formulation

The measured k-space data $\mathbf{y}$ are related to the unknown tissue parameters
$\mathbf{p}$ through the acquisition model

$$
\mathbf{y} \;=\; \mathbf{A}\,\mathbf{M}_{\boldsymbol{\phi}}\,\mathbf{Q}(\mathbf{p}) \;+\; \boldsymbol{\epsilon},
$$

where $\boldsymbol{\epsilon}$ accounts for measurement noise and each operator describes a
different aspect of the acquisition:

- $\mathbf{Q}$ — the exponential signal model, mapping the $P$ quantitative parameters
  $\mathbf{p}$ to a dynamic $\mathbf{Q}(\mathbf{p}) = (Q_{t_1}(\mathbf{p}),\dots,Q_{t_T}(\mathbf{p}))$,
  i.e. a sequence of $T$ images with time-varying contrast;
- $\mathbf{M}_{\boldsymbol{\phi}}$ — the motion operator, applying the displacement fields
  $\boldsymbol{\phi} = (\phi_1,\dots,\phi_T)$ time-wise to a dynamic via local spatial
  interpolation kernels; since data are acquired continuously over multiple cardiac cycles,
  these fields describe the full range of heart motion, including diastolic and systolic phases;
- $\mathbf{A}$ — the forward operator, combining coil sensitivity maps with the non-Cartesian
  Fourier transform defined by the chosen undersampling trajectory.

Recovering the motion-corrected parameters from $\mathbf{y}$ then corresponds to solving

$$
\mathbf{p}_{\text{MoCo}},\ \boldsymbol{\phi}_{\text{cardiac}}
\;=\;
\arg\min_{\mathbf{p},\,\boldsymbol{\phi}}\;
\big\|\mathbf{A}\,\mathbf{M}_{\boldsymbol{\phi}}\,\mathbf{Q}(\mathbf{p}) - \mathbf{y}\big\|_2^2 .
$$

Solving this directly is not tractable: the cardiac motion fields are unknown and hard to
recover without intermediate reconstruction steps, the parameter estimation is non-linear
because of $\mathbf{Q}$, and the k-space data are highly undersampled, making the
reconstruction itself ill-posed.

### Proposed joint optimization

Introducing the dynamic $\mathbf{x}$ and the low-rank coefficients $\mathbf{c}_j$ as auxiliary
variables (variable splitting), the problem is reformulated into two subproblems: **(a)** motion
estimation and **(b)** motion-corrected reconstruction and parameter estimation, which are
solved in alternation and stay coupled through the outer iterations:

$$
\begin{cases}
\boldsymbol{\theta}^{*} \in \arg\min_{\boldsymbol{\theta}} \ \mathcal{L}\big(\boldsymbol{\phi}_{\boldsymbol{\theta}},\boldsymbol{\phi}^{-1}_{\boldsymbol{\theta}},\mathbf{x}_{\text{m}},\mathbf{x}_{\text{r}}\big) & \text{(a)} \\[6pt]
\mathbf{p}^{*},\mathbf{x}^{*},\mathbf{c}_j^{*} \in \arg\min_{\mathbf{p},\mathbf{x},\mathbf{c}_j} \ \mathcal{F}\big(\mathbf{p},\mathbf{x},\mathbf{c}_j,\boldsymbol{\phi}_{\boldsymbol{\theta}^{*}}\big) & \text{(b)}
\end{cases}
$$

**(a) Motion estimation.** Forward and inverse motion fields are the output of a CNN,
$(\boldsymbol{\phi}_{\boldsymbol{\theta}}, \boldsymbol{\phi}^{-1}_{\boldsymbol{\theta}}) = \mathbf{u}_{\boldsymbol{\theta}}(\mathbf{x}_{\text{m}})$,
whose weights are optimized **zero-shot** (no training data, no pre-trained weights: the network
is fitted to this single scan) and **unsupervised** (no ground-truth motion or T1 is ever used),
by minimizing

$$
\mathcal{L}(\boldsymbol{\phi},\boldsymbol{\phi}^{-1},\mathbf{x}_{\text{m}},\mathbf{x}_{\text{r}})
=
\underbrace{\frac{1}{2}\sum_{t=1}^{T}\Big[\mathrm{MI}\big(\mathbf{M}_{\phi_t}x^{\text{r}}_t,\,x^{\text{m}}_t\big)+\mathrm{MI}\big(\mathbf{M}_{\phi_t^{-1}}x^{\text{m}}_t,\,x^{\text{r}}_t\big)\Big]}_{\text{bidirectional mutual-information loss}}

+\;\underbrace{\frac{1}{2}\big\|\mathbf{A}\mathbf{M}_{\boldsymbol{\phi}}\mathbf{x}_{\text{r}}-\mathbf{y}\big\|_2^2}_{\text{k-space data consistency}}
\;+\;\underbrace{\sum_{t=1}^{T}\Big(\lambda\|\nabla\phi_t\|_2^2+\mu\|\mathrm{div}\,\phi_t\|_2^2\Big)}_{\text{regularization on motion}}
$$

where $\mathbf{x}_{\text{m}}$ is the moving (uncorrected) dynamic and $\mathbf{x}_{\text{r}}$ a
synthetic motion-compensated reference, obtained by fitting the signal model to a subset of
frames (indicating diastole) and progressively updated across the outer iterations.

**(b) Motion-corrected reconstruction and parameter estimation.** Given the current motion
fields, the functional

$$
\mathcal{F}(\mathbf{p},\mathbf{x},\mathbf{c}_j,\boldsymbol{\phi})
=
\underbrace{\frac{1}{2}\big\|\mathbf{A}\mathbf{M}_{\boldsymbol{\phi}}\mathbf{x}-\mathbf{y}\big\|_2^2}_{\text{data consistency}}
+\underbrace{\frac{\lambda_1}{2}\big\|\mathbf{x}-\mathbf{Q}(\mathbf{p})\big\|_2^2}_{\text{signal-model penalty}}
+\underbrace{\frac{\lambda_2}{2}\sum_j\Big(\|\mathbf{R}_j\mathbf{x}-\mathbf{U}\mathbf{c}_j\|_2^2+\lambda_3\|\mathbf{c}_j\|_1\Big)}_{\text{SVD-based low-rank + sparsity}}
$$

is minimized with $\lambda_1,\lambda_2,\lambda_3>0$. Here $\mathbf{U}$ is a fixed temporal
basis obtained from an SVD of the initial dynamic, and $\mathbf{R}_j$ extracts the temporal
signal at voxel $j$.

### Alternating scheme

Each outer iteration updates one block at a time, keeping the others fixed:

| Step | Variable | Solver |
|:--:|:--:|---|
| 0 | $\mathbf{x}_{\text{r}}$ | reference update: L-BFGS fit of the signal model on an expanding, ECG-selected subset of frames |
| 1 | $\boldsymbol{\phi},\ \boldsymbol{\phi}^{-1}$ | zero-shot unsupervised registration network (MI matching + data consistency + smoothness) |
| 2 | $\mathbf{c}_j$ | soft-thresholding (closed form) |
| 3 | $\mathbf{x}$ | conjugate gradient on the quadratic subproblem |
| 4 | $\mathbf{p}$ | L-BFGS fit of the constrained signal model, on the $\lfloor 0.85\,T\rfloor$ frames with smallest motion amplitude |

After $N_{\text{iter}}$ outer iterations (here $N_{\text{iter}} = 4$, plus one initialization
iteration), the scheme returns the final motion-corrected parametric maps
$\mathbf{p}_{\text{MoCo}}$, in particular the T1 map.

<a id="1"></a>
## 1. Setup

Imports, device and hyper-parameters. The parameters are the same as in
[`main.py`](../scripts/main.py) so that the notebook stays in sync with the script.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import mrpro
import numpy as np
import torch
import zenodo_get

# the notebook runs from examples/notebooks: make the repository root and src/ importable
REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / 'src')]

from cardiac_motion.MotionEstimationWithInverse import MotionEstimationWithInverse
from operators.construct_recon_operators import compute_csm
from operators.MotionOperator import MotionOperator
from parameters_estimation.T1ParameterEstimator import T1ParameterEstimator
from regularization.svd import compute_svd, update_svd
from utils.MaskImgSize import MaskImgSize
from utils.rr_threshold_functions import find_valid_rr_threshold, update_rr_expansion_and_compute_rr_mask
from t1_colormap import cmap_t1  # T1 colormap used throughout the paper figures

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)

print(f'device: {device}')

In [ ]:
# data
ZENODO_RECORD_ID = '23388965'
N_LINES_PER_DYNAMIC = 16  # k-space lines per dynamic image
N_LINES_OVERLAP = 8  # lines shared by consecutive dynamics (sliding window)
CSM_SMOOTHING_WIDTH = 15

# acquisition parameter not stored in the k-space header (see section 2)
TAU = torch.tensor(0.028, dtype=torch.float32)  # inversion pulse -> first dynamic (s), protocol-dependent

# reconstruction
N_ITERATIONS = 4
CG_ITER_INITIAL = 8
CG_ITER_RECON = 5
CG_TOL = 1e-5
LAMBDA_1 = 0.1  # weight of the T1 model term
LAMBDA_2 = 0.05  # weight of the SVD (low-rank) term
LAMBDA_3 = 0.05 * 0.5  # L1 weight of the SVD coefficients

# output
SAVE_RESULTS = True
OUTPUT_DIR = Path.cwd()

# region of interest (heart), used for display, by the SVD prior and by the motion network.
mask_roi = MaskImgSize(h_start=70, w_start=68, h_width=100, w_width=100)

print(f'ROI mask              : {mask_roi.get_masked_shape()} at (h={mask_roi.h_start}, w={mask_roi.w_start})')
print(f'outer iterations      : {N_ITERATIONS} (+ initialization)')
print(f'lambda1 (signal model): {LAMBDA_1}')
print(f'lambda2 (SVD, l2)     : {LAMBDA_2}')
print(f'lambda3 (SVD, l1)     : {LAMBDA_3}')

<a id="2"></a>
## 2. Simulated data

**Data used here.** Numerical phantom generated with the XCAT framework: a short-axis
segmentation at the mid-ventricular level with the corresponding cardiac motion fields
$\boldsymbol{\phi}_{\text{xcat}}$ at 100 bpm. Tissue-specific quantitative parameters are
assigned to the segmentation and smoothed, giving the ground-truth maps
$\mathbf{p}_{\text{true}}$. Coil sensitivity maps estimated from an in-vivo scan and reoriented
to the phantom geometry define the acquisition operator $\tilde{\mathbf{A}}$, so that

$$
\mathbf{y}_{\text{simul}} \;=\; \tilde{\mathbf{A}}\,\mathbf{M}_{\boldsymbol{\phi}_{\text{xcat}}}\,\mathbf{Q}(\mathbf{p}_{\text{true}}) \;+\; \boldsymbol{\epsilon},
$$

with $\boldsymbol{\epsilon}$ complex Gaussian noise of standard deviation $10^{-3}$ of the
maximum k-space magnitude. Acquisition parameters match the in-vivo protocol: 2D golden-angle
radial sampling, continuous readout of 2.3 s after a single non-selective inversion pulse,
flip angle 5°, TE 2.03 ms, TR 5 ms, FOV 320 × 320 mm², in-plane resolution 1.3 × 1.3 mm²,
slice thickness 8 mm, and $T = 54$ images per dynamic.

The simulated k-space `kdata_simul.h5`, the cardiac-phase indices `rr_idx.pt` and the
ground-truth maps `true_maps.pt` are downloaded from Zenodo. The k-space was generated as follows:

1. the XCAT ground-truth, piecewise-constant maps $(M_0, \alpha, T_1)$ are pushed through the transient
   steady-state signal model to get the motion-free dynamics $x^\text{true}$;
2. the *true* XCAT motion fields warp them into moving dynamics;
3. the moving dynamics are sampled with the real acquisition operator (trajectory and
   coil maps taken from an in-vivo scan) and complex Gaussian noise is added.

Quantities used below:

- `kdata_simul_dynamic` — simulated k-space, split into one dynamic per time frame. Its
  header also provides the timings of the signal model $\mathbf{Q}$, exactly as for acquired
  data: the acquisition time stamps (from which the sampling time of each dynamic is
  computed), the repetition time and the nominal flip angle;
- `TAU` — the delay between the inversion pulse and the first dynamic. The header time stamps
  are not referenced to the inversion pulse, so this offset is not available from the data and
  is set by hand in [section 1](#1) (28 ms here). It depends on the sequence and protocol, so
  it must match the scan being reconstructed;
- `rr_idx` — cardiac-phase index of each dynamic: the RR interval (time between two
  consecutive R-peaks of the ECG) is divided into bins, and `rr_idx[t]` is the bin in which
  dynamic $t$ was acquired (here 0–10). Dynamics with the same index show the heart in
  approximately the same phase, which is used in [section 5](#5) to build the
  motion-compensated reference. For the phantom it is provided with the data; for patients
  and volunteers it is computed from the ECG recorded during the scan;
- `p_true` — ground-truth, piecewise-constant $(M_0, \alpha[\deg], T_1[s])$ of shape `(3, 240, 240)`,
  already in the scanner orientation; only used for the final figure in [section 9](#9).

In [ ]:
# KData.from_file needs a path, so the download goes to a temporary folder that is deleted after loading.
with tempfile.TemporaryDirectory() as tmp_dir:
    zenodo_get.download(
        record=ZENODO_RECORD_ID,
        output_dir=tmp_dir,
        file_glob=('kdata_simul.h5', 'rr_idx.pt', 'true_maps.pt'),
        timeout=120,  # the default of 15 s is too short behind slow proxies
        retry_attempts=3,
    )
    kdata_simul = mrpro.data.KData.from_file(
        os.path.join(tmp_dir, 'kdata_simul.h5'),
        mrpro.data.traj_calculators.KTrajectoryIsmrmrd(),
    ).to(device)
    # cardiac-phase (RR-interval) index of each dynamic
    rr_idx = torch.load(os.path.join(tmp_dir, 'rr_idx.pt'))
    # ground-truth maps (M0, flip angle [deg], T1 [s])
    p_true = torch.load(os.path.join(tmp_dir, 'true_maps.pt'), map_location='cpu')

# split k-space data into dynamics
sorted_idx = kdata_simul.header.acq_info.acquisition_time_stamp.squeeze().argsort()
split_idx = mrpro.utils.split_idx(sorted_idx, N_LINES_PER_DYNAMIC, N_LINES_OVERLAP)
kdata_simul_dynamic = kdata_simul[..., split_idx, :]

print(f'simulated k-space : {tuple(kdata_simul_dynamic.data.shape)}')
print(f'ground-truth maps : {tuple(p_true.shape)}')
print(f'dynamics          : {len(rr_idx)}   (RR index range: {rr_idx.min().item()}-{rr_idx.max().item()})')

<a id="3"></a>
## 3. Operators and initial reconstruction

The pieces of the forward model are built from the k-space data (coil sensitivities with `compute_csm`):

- `acquisition_op` $= \mathbf{A}$ — coil sensitivities + non-uniform FFT along the trajectory,
- `dcf_op` $= \mathbf{W}$ — density compensation, `dcf_sqrt_op` $= \mathbf{W}^{1/2}$,
- `AHWA_op` $= \mathbf{A}^H \mathbf{W} \mathbf{A}$ — the normal operator used inside CG.

Solving $\mathbf{A}^H \mathbf{W} \mathbf{A}\, \mathbf{x} = \mathbf{A}^H \mathbf{W} \mathbf{y}$ with a few CG iterations gives $\mathbf{x}_\text{m}$: the
undersampled **moving** dynamics. This is the starting point of the
alternating optimization.

In [ ]:
# compute acquisition operator for reconstruction
csm_op, img_size = compute_csm(kdata_simul, smoothing_width=CSM_SMOOTHING_WIDTH, device=device)
dcf = mrpro.data.DcfData.from_traj_voronoi(kdata_simul_dynamic.traj).to(device)
dcf_sqrt_op = mrpro.operators.DensityCompensationOp(torch.sqrt(dcf.data))
dcf_op = dcf.as_operator()
fourier_op = mrpro.operators.FourierOp.from_kdata(kdata_simul_dynamic).to(device)

acquisition_op = fourier_op @ csm_op
AHWA_op = (dcf_sqrt_op @ acquisition_op).gram

(rhs_no_motion,) = (acquisition_op.H @ dcf_op)(kdata_simul_dynamic.data)

# initial (not motion-corrected) reconstruction of the dynamics
with torch.no_grad():
    (x_moving,) = mrpro.algorithms.optimizers.cg(
        AHWA_op,
        rhs_no_motion,
        initial_value=rhs_no_motion,
        max_iterations=CG_ITER_INITIAL,
        tolerance=CG_TOL,
    )
torch.cuda.empty_cache()

num_dynamics = x_moving.shape[0]

print(f'x_moving     : {tuple(x_moving.shape)}')
print(f'num_dynamics : {num_dynamics}')
print(f'image size   : {img_size}')

<a id="4"></a>
## 4. T1 estimator and No MoCo baseline

`T1ParameterEstimator` wraps the transient steady-state inversion-recovery model
$\mathbf{Q}(\mathbf{p})$ together with a `ConstraintsOp` that keeps $T_1$ and $\alpha$ in physical
ranges, and fits $\mathbf{p}$ by L-BFGS.

Fitted directly to the moving dynamics $\mathbf{x}_{\text{m}}$, over all $T$ frames, it returns the
baseline T1 map without any motion correction.

Since the acquisition is continuous, these frames span the full cardiac motion: voxels that
move across tissue boundaries are fitted from a mix of signals, and the T1 map degrades
exactly where motion is largest — the myocardial borders and the papillary muscles.

The same map is reused in the final comparison of [section 9](#9).

In [ ]:
# timings and flip angle from the k-space header, TAU set by hand
t1_parameter_estimator = T1ParameterEstimator(
    kdata=kdata_simul_dynamic,
    tau=TAU,
    opt_type='lbfgs',
)

# Baseline: signal model fitted directly to the moving (uncorrected) dynamics, all T frames.
p_no_moco, _ = t1_parameter_estimator.fit(x_moving)

T1_CLIM = (0, 2000)  # ms
# the reconstructed maps are transposed with respect to the ground truth: transpose them before cropping
t1_no_moco = mask_roi.apply(p_no_moco[-1].squeeze().abs().mT).detach().cpu().numpy() * 1e3


def prepare_image(img):
    return np.rot90(np.flipud(img), k=-1)  # flip vertically, then rotate 90 deg clockwise


fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(prepare_image(t1_no_moco), cmap=cmap_t1, clim=T1_CLIM)
ax.set_title('No MoCo')
ax.axis('off')
fig.colorbar(im, ax=ax, fraction=0.046, label='T1 [ms]')
plt.show()

<a id="5"></a>
## 5. Motion-compensated reference

To guide the registration we need a **motion-compensated target**. `find_valid_rr_threshold`
takes the cardiac phase of the first dynamic, `rr_idx[0]`, as base and selects the dynamics
whose `rr_idx` lies within a few bins of it (offsets `[0, 1, -1, 2]`, wrapping around the
heartbeat). If the fit on this subset produces NaNs, the window is widened by one bin at a
time (`-2`, `3`, …) until it succeeds. Fitting $\mathbf{p}^{(0)}_{\text{r}}$ on that
subset and re-synthesizing $\mathbf{Q}(\mathbf{p}^{(0)}_\text{r})$ over all time points gives an initial motion-compensated
reference dynamic $\mathbf{x}^{(0)}_{\text{r}}$ that acts as the fixed image for registration.

In [ ]:
# initial T1 parameter fitting with ECG-based mask for motion-compensated references.
p_0_ref, x_0_ref, rr_expansion_offsets = find_valid_rr_threshold(
    rr_idx,
    x_moving,
    t1_parameter_estimator,
)
if p_0_ref is None or x_0_ref is None:
    raise RuntimeError('Could not find valid RR threshold.')

# store rr_idx expansion pattern for iterative updates.
base_rr_idx = rr_idx[0].item()
current_rr_expansion = rr_expansion_offsets.copy()

print(f'reference built with RR expansion {rr_expansion_offsets} (base index {base_rr_idx})')

<a id="6"></a>
## 6. Motion estimation

`MotionOperator` holds the forward warp $\mathbf{M}_{\boldsymbol{\phi}}$ and its inverse $\mathbf{M}_{\boldsymbol{\phi}^{-1}}$ as
`GridSamplingOp`s; it is initialized with zero displacement and is updated in place by
the registration network.

`MotionEstimationWithInverse` is a small U-Net that predicts, for every dynamic, the
displacement field to the reference phase **and** its inverse. It is trained from
scratch on this subject only (*zero-shot*) and without ground-truth motion
(*unsupervised*), minimizing the loss function

$$\arg\min_{\boldsymbol{\theta}} \ \mathcal{L}\big(\boldsymbol{\phi}_{\boldsymbol{\theta}},\boldsymbol{\phi}^{-1}_{\boldsymbol{\theta}},\mathbf{x}_{\text{m}},\mathbf{x}_{\text{r}}\big)$$

`mask_roi` restricts the loss to the cardiac ROI.

In [ ]:
motion_op = MotionOperator(
    times=num_dynamics,
    img_size=img_size,
    device=device,
)
motion_estimation = MotionEstimationWithInverse(
    kdata_dynamic=kdata_simul_dynamic,
    acquisition_op=acquisition_op,
    dcf_sqrt_op=dcf_sqrt_op,
    motion_op=motion_op,
    img_size=img_size,
    times=num_dynamics,
    mask_roi=mask_roi,
    device=device,
)

In [ ]:
# initial cardiac motion fields estimation.
motion_estimation.train_net(
    x_moving.abs(),
    x_0_ref,
    episodes=40,
    steps=20,
    full_loss_phi=35,
)
torch.cuda.empty_cache()
print('\ninitial motion fields estimated')

<a id="7"></a>
## 7. Initialization (iteration 0)

Three things are initialized before the loop starts:

**$\mathbf{x}^{(0)}$** — apply the inverse motion to the moving dynamics, $\mathbf{x}^{(0)} = \mathbf{M}_{\boldsymbol{\phi}^{-1}} \mathbf{x}_\text{m}$,
so that all frames are brought back to the common reference phase.

**$\mathbf{p}^{(0)}$, $\mathbf{Q}(\mathbf{p}^{(0)})$** — fit the signal model on $\mathbf{x}^{(0)}$. Even after motion
correction, the frames acquired near peak systole are the least reliable, so
`select_sinusoidal_mask` fits a sinusoid to the mean motion amplitude over time and
drops the top 15 % (`percentage=0.85`, the default of `select_sinusoidal_mask`) of the curve.

**SVD basis $\mathbf{U}$** — a temporal low-rank basis computed on the ROI of $\mathbf{x}^{(0)}$.

In [ ]:
# compute current registered dynamics and update current x with it
(x_registered,) = motion_op.M_op_inv(x_moving)
x_registered = x_registered.detach()
x_current = x_registered
print(f'x_current: {tuple(x_current.shape)}')

# compute mask selection from sinusoidal fit to exclude 15% systolic frames
optimal_phi = MotionOperator.normalized_grid_to_field(motion_op.M_op.grid)
mask_85_percent = T1ParameterEstimator.select_sinusoidal_mask(
    optimal_phi.detach().cpu(),
)
if mask_85_percent is not None:
    print(f'dynamics kept for fitting: {mask_85_percent.sum()}/{len(mask_85_percent)}')

# initial motion-corrected T1 parameter fitting
p, qp = t1_parameter_estimator.fit(x_current, mask=mask_85_percent)

# initial SVD-based computation
x_reg_svd, U_op, _ = compute_svd(x_current, mask_roi)
x_reg_svd = x_reg_svd.detach().unsqueeze(1).unsqueeze(1)
torch.cuda.empty_cache()
print(f'p: {tuple(p.shape)} | qp: {tuple(qp.shape)} | x_reg_svd: {tuple(x_reg_svd.shape)}')

print('Initialization completed.')

<a id="8"></a>
## 8. Alternating optimization loop

Each outer iteration cycles through the three blocks.

**(a) Motion fields $\boldsymbol{\phi}$, $\boldsymbol{\phi}^{-1}$.** The cardiac-phase window used to build the reference is
widened by one RR bin, a new $\mathbf{x}_\text{r}$ is fitted on it, and the registration network
is fine-tuned against it. Widening the window means more dynamics — hence more inversion
times and a better-conditioned T1 fit — which is only possible because the images are by
now partly motion-corrected. If the wider fit produces NaNs we fall back to the previous
reference.

**(b) Motion-corrected dynamic $\mathbf{x}$.** With $\boldsymbol{\phi}$ and $\mathbf{p}$ fixed, the objective is quadratic in $\mathbf{x}$ and its
normal equation

$$
\big(\mathbf{M}_{\boldsymbol{\phi}}^{H}\mathbf{A}^{H}\mathbf{W}\mathbf{A}\mathbf{M}_{\boldsymbol{\phi}} + \lambda_1 \mathrm{Id} + \lambda_2 \mathrm{Id}\big)\, \mathbf{x}
= \mathbf{M}_{\boldsymbol{\phi}}^{H}\mathbf{A}^{H}\mathbf{W}\mathbf{y} + \lambda_1\, \mathbf{Q}(\mathbf{p}) + \lambda_2\, \mathbf{U}\mathbf{U}^{H}\mathbf{x}
$$

is solved with a few CG iterations, warm-started from the previous $\mathbf{x}$. The SVD term on
the right-hand side is refreshed first by `update_svd` (soft-thresholding with weights
`LAMBDA_3` / `LAMBDA_2`).

**(c) Parameters $\mathbf{p}$.** Refit the signal model on the updated $\mathbf{x}$, again excluding the
systolic frames.

In [ ]:
print('Starting iterative loop.')
# alternate motion estimation, SVD regularization, motion-corrected CG reconstruction
# and motion-corrected T1 estimation.
for iteration in range(1, N_ITERATIONS + 1):
    print(f'\n{"-" * 40}\nIteration {iteration}/{N_ITERATIONS}')

    # expand the rr_idx selection and compute the mask of the reference dynamics
    mask_ref, current_rr_expansion = update_rr_expansion_and_compute_rr_mask(
        rr_idx,
        base_rr_idx,
        current_rr_expansion,
    )
    n_ref = mask_ref.sum().item()
    print(f'  x_ref : RR expansion {current_rr_expansion} -> {n_ref} dynamics '
          f'({n_ref / mask_ref.numel() * 100:.1f}%)')

    if not mask_ref.any():
        p_ref_updated = p_0_ref
        x_ref_updated = x_0_ref
    else:
        # try fitting with current mask
        p_ref_updated, x_ref_updated = t1_parameter_estimator.fit(
            x_current,
            mask=mask_ref,
        )
        # if NaN, fall back to previous values and don't expand further
        if torch.isnan(x_ref_updated).any():
            print('  x_ref contains NaN -> keeping the previous reference')
            p_ref_updated = p_0_ref
            x_ref_updated = x_0_ref

    # update cardiac motion fields estimation
    motion_estimation.train_net(
        x_moving.abs(),
        x_ref_updated.detach(),
        episodes=30,
        steps=20,
        full_loss_phi=25,
    )
    torch.cuda.empty_cache()
    print(f'\nphi^({iteration}) updated.')  # train_net prints with a carriage return

    # update SVD regularizer with soft-thresholding
    x_reg_svd, _ = update_svd(
        x_current,
        U_op,
        weight_l1=LAMBDA_3,
        weight_l2=LAMBDA_2,
    )
    print(f'c_j^({iteration}) updated.')

    with torch.no_grad():
        # motion-corrected image reconstruction with updated motion fields and SVD regularization.
        # solve the quadratic subproblem with CG.
        H = lambda y: (
            motion_op.adjoint(AHWA_op(motion_op.forward(y)[0])[0])[0]
            + LAMBDA_1 * y
            + LAMBDA_2 * y,
        )

        (b_0,) = motion_op.adjoint(rhs_no_motion)
        b = (
            b_0
            + LAMBDA_1 * qp
            + LAMBDA_2 * x_reg_svd
        )

        (x_next,) = mrpro.algorithms.optimizers.cg(
            operator=H,
            right_hand_side=b,
            initial_value=x_current,
            max_iterations=CG_ITER_RECON,
            tolerance=CG_TOL,
        )
    x_current = x_next.detach()
    torch.cuda.empty_cache()
    print(f'x^({iteration}) updated.')

    # initialize the parametric maps for further fitting
    p_init = p_ref_updated.detach()

    # automatic extraction of systolic frames
    optimal_phi = MotionOperator.normalized_grid_to_field(motion_op.M_op.grid)
    mask_85_percent = T1ParameterEstimator.select_sinusoidal_mask(
        optimal_phi.detach().cpu(),
    )

    # motion-corrected T1 parameter fitting on updated image, using chosen initialization.
    p, qp = t1_parameter_estimator.fit(
        x_current,
        mask=mask_85_percent,
        initial_p=p_init,
    )
    torch.cuda.empty_cache()
    print(f'p^({iteration}) updated.')


if SAVE_RESULTS:
    torch.save(p, OUTPUT_DIR / 'p_moco.pt')
    print(f'Results saved to {OUTPUT_DIR}')

print('Finished processing.')

<a id="9"></a>
## 9. Final T1 map

`p` stacks $(M_0, \alpha, T_1)$, so `p[-1]` is the T1 map in **seconds**; it is converted
to milliseconds for display and cropped to the cardiac ROI.

Three maps are shown side by side:

- **No MoCo** — the uncorrected baseline of [section 4](#4);
- **Proposed** — the motion-corrected T1 map returned by the iterative scheme;
- **Reference** — the ground truth from $\mathbf{p}_{\text{true}}$, i.e. the tissue-specific
  values assigned to the XCAT segmentation and smoothed with a Gaussian kernel to soften the
  sharp, unrealistic tissue boundaries.

In [ ]:
t1_estimated = mask_roi.apply(p[-1].squeeze().abs().mT).detach().cpu().numpy() * 1e3
t1_ground_truth = mask_roi.apply(p_true[-1].abs()).numpy() * 1e3

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5))
for ax, img, title in zip(axes, [t1_no_moco, t1_estimated, t1_ground_truth], ['No MoCo', 'Proposed', 'Reference']):
    im = ax.imshow(prepare_image(img), cmap=cmap_t1, clim=T1_CLIM)
    ax.set_title(title)
    ax.axis('off')

fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.03, label='T1 [ms]')
fig.suptitle('XCAT Simulated Data')
plt.show()